In [1]:
import time
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import SGDClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

In [2]:
DATASET_NAME = "openSMILE 3.0s"

DATASET_PATH = (
    "/Users/bhavaykhatri/Desktop/embeddings/openSMILE/"
    "singBAP_dataset_opensmile-compare-2016_3.0s.parquet"
)

df = pd.read_parquet(DATASET_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Shape: (4116, 13)
Columns: ['condition', 'experience', 'extractor', 'filename', 'filepath', 'frame_index', 'phonation', 'scale', 'singer', 'take', 'embedding', 'embedding_dtype', 'embedding_shape']


,condition,experience,extractor,filename,filepath,frame_index,phonation,scale,singer,take,embedding,embedding_dtype,embedding_shape
0,after_instruction,inexperienced,opensmile-compare-2016,inex-1-after_instruction-glissando-1-mic-audio,/home/suvihaara/Documents/PhD/DATA/VTE/SingBAP...,0,,glissando,INEX-1,1,b'\xd5\x81\xa9=\xbbb#?\x00\x00\x00\x00x*S=\xa4...,<f4,[6373]
1,after_instruction,inexperienced,opensmile-compare-2016,inex-1-after_instruction-glissando-2-mic-audio,/home/suvihaara/Documents/PhD/DATA/VTE/SingBAP...,0,,glissando,INEX-1,2,b'<*\xac=s\xcc^?\xda\xbb5=\xe4\xe4v=o\xbd\x8a=...,<f4,[6373]
2,after_instruction,inexperienced,opensmile-compare-2016,inex-1-after_instruction-glissando-2-mic-audio,/home/suvihaara/Documents/PhD/DATA/VTE/SingBAP...,1,,glissando,INEX-1,2,b'r\xf8\xb3=]\xb1\xd1=T \x7f?\x18\x1f{=M\xf3\x...,<f4,[6373]
3,after_instruction,inexperienced,opensmile-compare-2016,inex-1-after_instruction-glissando-3-mic-audio,/home/suvihaara/Documents/PhD/DATA/VTE/SingBAP...,0,,glissando,INEX-1,3,b'B\xb5\x15>\xdeZ8?\x1f\xac\xdf;\x10\xdb\xad=\...,<f4,[6373]
4,after_instruction,inexperienced,opensmile-compare-2016,inex-1-after_instruction-glissando-3-mic-audio,/home/suvihaara/Documents/PhD/DATA/VTE/SingBAP...,1,,glissando,INEX-1,3,b'\xedA\xb3=V\xc6\x99=T \x7f?k\xec\x98= \n\xda...,<f4,[6373]


In [3]:
TARGET_CLASSES = [
    "correct",
    "arched_back",
    "hunched_back",
    "sideways",
    "chest_breathing",
    "over_articulation",
    "under_articulation",
]

df = df[
    df["experience"].isin(
        ["intermediate", "professional"]
    )
].copy()

df = df[
    df["condition"].isin(TARGET_CLASSES)
].copy()

print("Filtered shape:", df.shape)
print(df["condition"].value_counts())
print(df["experience"].value_counts())

Filtered shape: (3438, 13)
condition
correct               638
hunched_back          556
sideways              521
chest_breathing       512
over_articulation     411
arched_back           405
under_articulation    395
Name: count, dtype: int64
experience
intermediate    2794
professional     644
Name: count, dtype: int64


In [4]:
def decode_embedding(value):
    if isinstance(
        value,
        (bytes, bytearray, memoryview),
    ):
        return np.frombuffer(
            value,
            dtype=np.float32,
        ).copy()

    return np.asarray(
        value,
        dtype=np.float32,
    ).reshape(-1)

In [5]:
X = np.vstack(
    df["embedding"].apply(decode_embedding)
)

y = df["condition"].astype(str).to_numpy()

groups = df["filename"].astype(str).to_numpy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique recordings:", len(np.unique(groups)))

X shape: (3438, 6373)
y shape: (3438,)
Unique recordings: 3046


In [6]:
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups,
    )
)

X_train = X[train_idx]
X_test = X[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

train_groups = groups[train_idx]
test_groups = groups[test_idx]

print("Train:", X_train.shape)
print("Test:", X_test.shape)

print(
    "Shared recordings:",
    len(
        set(train_groups)
        & set(test_groups)
    ),
)

Train: (2750, 6373)
Test: (688, 6373)
Shared recordings: 0


In [7]:
MODELS = {
    "MLP": make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(256, 128),
            early_stopping=True,
            max_iter=300,
            random_state=42,
        ),
    ),

    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(
            n_neighbors=15,
            metric="cosine",
            n_jobs=-1,
        ),
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),

    "Linear SVM": make_pipeline(
    StandardScaler(),
    SGDClassifier(
        loss="hinge",
        class_weight="balanced",
        max_iter=2000,
        tol=1e-3,
        random_state=42,
        n_jobs=-1,
    ),
),
}

In [8]:
results = []

for model_name, base_model in MODELS.items():

    print(f"Training {model_name}...")

    model = clone(base_model)

    start = time.time()

    model.fit(
        X_train,
        y_train,
    )

    predictions = model.predict(
        X_test
    )

    elapsed = time.time() - start

    results.append({
        "Embedding": DATASET_NAME,
        "Model": model_name,

        "Accuracy": round(
            accuracy_score(
                y_test,
                predictions,
            ),
            4,
        ),

        "Balanced Accuracy": round(
            balanced_accuracy_score(
                y_test,
                predictions,
            ),
            4,
        ),

        "Macro F1": round(
            f1_score(
                y_test,
                predictions,
                average="macro",
                zero_division=0,
            ),
            4,
        ),

        "Train/Eval Time (s)": round(
            elapsed,
            2,
        ),

        "Samples": len(y),
        "Features": X.shape[1],
    })

Training MLP...
Training KNN...
Training Random Forest...
Training Linear SVM...


In [9]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Macro F1",
    ascending=False,
).reset_index(drop=True)

results_df

,Embedding,Model,Accuracy,Balanced Accuracy,Macro F1,Train/Eval Time (s),Samples,Features
0,openSMILE 3.0s,MLP,0.4680,0.4742,0.4763,41.23,3438,6373
1,openSMILE 3.0s,Random Forest,0.4738,0.4880,0.4760,12.77,3438,6373
2,openSMILE 3.0s,Linear SVM,0.3663,0.3688,0.3730,36.64,3438,6373
3,openSMILE 3.0s,KNN,0.3169,0.3140,0.3127,1.53,3438,6373


In [10]:
results_df.to_csv(
    "opensmile_3.0s_baseline.csv",
    index=False,
)

print("Saved: opensmile_3.0s_baseline.csv")

Saved: opensmile_3.0s_baseline.csv
